In [5]:
# Standard library imports for core functionality
import json  # Used for reading, writing, and parsing JSON file structures
import math  # Used for ceiling function (math.ceil) in P95 calculation
import os  # Standard OS interface utilities
import random  # Generates random choices, integers, and floats for dummy data
import shutil  # Handles directory cleanup (shutil.rmtree)
import time  # Provides precise time measurement (time.perf_counter) for performance profiling
from datetime import date, timedelta  # Generates ISO date strings and performs date arithmetic
from pathlib import Path  # Object-oriented filesystem path management

# CONFIGURATION & CONSTANTS
BASE_DIR = Path("/tmp/flights")  # Output root directory for generated files
NUM_FILES = 5000  # Total number of JSON files to generate
CITY_POOL_SIZE = 150  # Number of unique city names (K = [100, 200])
MIN_RECORDS_PER_FILE = 50  # Minimum flight records per file (Range [50, 100])
MAX_RECORDS_PER_FILE = 100  # Maximum flight records per file (Range [50, 100])
DIRTY_PROBABILITY = 0.008  # Probability of corrupting a record with NULLs (L = [0.5%, 1.0%])

# Generate a mock pool of city names: ['City_001', 'City_002', ..., 'City_150']
CITIES = [f"City_{i:03d}" for i in range(1, CITY_POOL_SIZE + 1)]


def generate_flight_record(origin_city: str) -> dict:
    """Generates a single dictionary flight record with realistic attributes.

    Randomly replaces 1-3 field values with None (NULL) based on
    DIRTY_PROBABILITY.
    """
    # Select a destination city distinct from the origin city
    destination_city = random.choice([c for c in CITIES if c != origin_city])

    # Generate a random flight date within the last 365 days in ISO format (YYYY-MM-DD)
    random_days = random.randint(0, 365)
    flight_date = (date.today() - timedelta(days=random_days)).isoformat()

    # Construct clean record payload
    record = {
        "date": flight_date,
        "origin_city": origin_city,
        "destination_city": destination_city,
        "flight_duration_secs": random.randint(
            1800, 43200
        ),  # Duration between 30 mins and 12 hours
        "#_of_passengers_on_board": random.randint(
            20, 350
        ),  # Passenger payload range
    }

    # Inject NULL values into random fields based on probability check
    if random.random() < DIRTY_PROBABILITY:
        # Select 1 to 3 random keys in the record to clear to None
        fields_to_nullify = random.sample(
            list(record.keys()), k=random.randint(1, 3)
        )
        for field in fields_to_nullify:
            record[field] = None

    return record


def run_phase_1():
    """Phase 1: Data Generation.

    Creates structured target directories and populates them with ~5000 JSON
    files containing generated flight records.
    """
    print("--- Phase 1: Starting Data Generation ---")

    # Wipe existing execution directory to prevent stale data or directory/file path conflicts
    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR)

    # Recreate clean root directory structure
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    # Format current date string for path hierarchy requirements (%MM-YY%)
    current_mmyy = date.today().strftime("%m-%y")

    # Generate total batch size of JSON files
    for i in range(NUM_FILES):
        # Pick a random origin city for this batch
        origin_city = random.choice(CITIES)

        # Construct file name adhering to pattern: /tmp/flights/%MM-YY%-%origin_city%-flights_%index%.json
        file_path = (
            BASE_DIR / f"{current_mmyy}-{origin_city}-flights_{i+1}.json"
        )

        # Generate M records for this specific file (between 50 and 100)
        num_records = random.randint(MIN_RECORDS_PER_FILE, MAX_RECORDS_PER_FILE)
        file_data = [
            generate_flight_record(origin_city) for _ in range(num_records)
        ]

        # Write generated records to JSON file
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(file_data, f, indent=2)

    print(
        f"Phase 1 Complete: Generated {NUM_FILES} files directly in {BASE_DIR}.\n"
    )


def run_phase_2():
    """Phase 2: Data Cleaning & Analytical Processing.

    Reads generated JSON payloads, filters out dirty records containing NULL
    values, and computes arrival durations, P95, and passenger balances.
    """
    print("--- Phase 2: Starting Data Analysis & Cleaning ---")

    # Start high-precision timer to profile Phase 2 execution speed
    start_time = time.perf_counter()

    # Track operational processing metrics
    total_records = 0
    dirty_records = 0

    # In-memory aggregation data structures
    dest_durations = {}  # Map: destination_city -> list of durations (secs)
    dest_passengers = {}  # Map: destination_city -> total incoming passengers
    passenger_balance = {
        city: 0 for city in CITIES
    }  # Map: city -> net balance (arrivals - departures)

    # Safely retrieve all JSON file paths (filtering out any directory matches)
    json_files = [p for p in BASE_DIR.glob("**/*.json") if p.is_file()]

    # Iteratively parse and process each JSON batch file
    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                records = json.load(f)
            except json.JSONDecodeError:
                continue  # Skip unparseable files if encountered

        # Evaluate individual flight records
        for record in records:
            total_records += 1

            # Identify "dirty" records: drop record if ANY key contains a None value
            if any(value is None for value in record.values()):
                dirty_records += 1
                continue  # Discard record from analytical metrics

            # Extract validated fields
            origin = record["origin_city"]
            dest = record["destination_city"]
            duration = record["flight_duration_secs"]
            passengers = record["#_of_passengers_on_board"]

            # Aggregate flight duration list for destination
            dest_durations.setdefault(dest, []).append(duration)

            # Aggregate total arrival passenger volume
            dest_passengers[dest] = dest_passengers.get(dest, 0) + passengers

            # Compute net passenger migration flow
            passenger_balance[origin] = (
                passenger_balance.get(origin, 0) - passengers
            )
            passenger_balance[dest] = (
                passenger_balance.get(dest, 0) + passengers
            )

    # Sort destination cities by incoming volume and isolate Top 25
    top_25_dests = sorted(
        dest_passengers.items(), key=lambda x: x[1], reverse=True
    )[:25]

    top_25_metrics = []
    for dest, _ in top_25_dests:
        # Sort duration list in ascending order to evaluate statistical metrics
        durations = sorted(dest_durations[dest])
        n = len(durations)

        # Mean flight duration
        avg_dur = sum(durations) / n if n > 0 else 0

        # Nearest-rank 95th Percentile calculation: index = ceil(0.95 * N) - 1
        p95_index = math.ceil(0.95 * n) - 1
        p95_dur = durations[max(0, p95_index)] if n > 0 else 0

        top_25_metrics.append(
            {
                "city": dest,
                "total_passengers": dest_passengers[dest],
                "avg_duration_sec": round(avg_dur, 2),
                "p95_duration_sec": p95_dur,
            }
        )

    # Extract cities with maximum and minimum net passenger balance
    max_balance_city = max(passenger_balance.items(), key=lambda x: x[1])
    min_balance_city = min(passenger_balance.items(), key=lambda x: x[1])

    # Calculate total processing runtime in milliseconds
    elapsed_ms = (time.perf_counter() - start_time) * 1000

    # Output formatted report
    print("================ ANALYSIS RESULTS ================")
    print(f"Total Records Processed : {total_records}")
    print(f"Total Dirty Records     : {dirty_records}")
    print(f"Total Runtime           : {elapsed_ms:.2f} ms\n")

    print("Top 25 Destination Cities Metrics:")
    print(
        f"{'City':<12} | {'Passengers':<12} | {'Avg Duration (s)':<18} | {'P95 Duration (s)':<16}"
    )
    print("-" * 65)
    for m in top_25_metrics:
        print(
            f"{m['city']:<12} | {m['total_passengers']:<12} | {m['avg_duration_sec']:<18} | {m['p95_duration_sec']:<16}"
        )

    print("\nPassenger Balance Analysis:")
    print(
        f"Max Remaining Passengers : {max_balance_city[0]} ({max_balance_city[1]:+} passengers)"
    )
    print(
        f"Min Remaining Passengers : {min_balance_city[0]} ({min_balance_city[1]:+} passengers)"
    )
    print("==================================================")


# Entry point execution
if __name__ == "__main__":
    run_phase_1()
    run_phase_2()

--- Phase 1: Starting Data Generation ---
Phase 1 Complete: Generated 5000 files directly in /tmp/flights.

--- Phase 2: Starting Data Analysis & Cleaning ---
================ ANALYSIS RESULTS ================
Total Records Processed : 374467
Total Dirty Records     : 2977
Total Runtime           : 599.87 ms

Top 25 Destination Cities Metrics:
City         | Passengers   | Avg Duration (s)   | P95 Duration (s)
-----------------------------------------------------------------
City_014     | 485662       | 22256.79           | 41097           
City_083     | 484935       | 22157.65           | 41222           
City_084     | 483791       | 22815.4            | 41292           
City_105     | 477268       | 22445.21           | 41216           
City_050     | 475819       | 22658.07           | 41209           
City_030     | 475190       | 22442.58           | 41049           
City_103     | 474684       | 22526.39           | 40835           
City_060     | 474206       | 22567.6       